# 10 — External Domain-Shift Evaluation on PlantDoc

This notebook evaluates the final PlantVillage-trained EfficientNet-B0 classifier on PlantDoc test images.

- Training domain: PlantVillage controlled leaf images.
- External domain: PlantDoc field-like leaf images.
- Evaluation protocol: zero-shot external test evaluation.
- PlantDoc training images are not used in this notebook.
- Only semantically compatible PlantDoc classes are included in supervised metrics.

The purpose is to quantify domain shift, not to maximize PlantDoc accuracy through retraining.

In [ ]:
from pathlib import Path
import json
import os
import random
import re
import time

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from PIL import Image, ImageFile

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import models, transforms

from sklearn.metrics import (
    accuracy_score,
    balanced_accuracy_score,
    classification_report,
    confusion_matrix,
    f1_score,
)

ImageFile.LOAD_TRUNCATED_IMAGES = True

SEED = 42

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)

if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

IMAGE_SIZE = 224
BATCH_SIZE = 32 if DEVICE.type == "cuda" else 8
NUM_WORKERS = 2 if DEVICE.type == "cuda" else 0

INPUT_ROOT = Path("/kaggle/input")
WORKING_ROOT = Path("/kaggle/working")

# User-supplied PlantDoc dataset location.
PLANTDOC_ROOT = Path(
    "/kaggle/input/datasets/nirmalsankalana/plantdoc-dataset"
)

# The PlantDoc external evaluation split.
EVAL_ROOT = PLANTDOC_ROOT / "test"

RESULTS_DIR = (
    WORKING_ROOT
    / "results"
    / "notebook_10_external_domain_shift_plantdoc"
)

RESULTS_DIR.mkdir(parents=True, exist_ok=True)

print(f"Device: {DEVICE}")
print(f"PlantDoc root: {PLANTDOC_ROOT}")
print(f"PlantDoc root exists: {PLANTDOC_ROOT.exists()}")
print(f"PlantDoc test exists: {EVAL_ROOT.exists()}")
print(f"Results directory: {RESULTS_DIR}")

In [ ]:
assert PLANTDOC_ROOT.exists(), (
    f"PlantDoc dataset was not found at:\n{PLANTDOC_ROOT}\n\n"
    "Check the Kaggle Input panel and confirm the mounted dataset path."
)

assert EVAL_ROOT.exists(), (
    f"PlantDoc test directory was not found at:\n{EVAL_ROOT}\n\n"
    "Expected structure: plantdoc-dataset/test/<class-folder>/<image>.jpg"
)

IMAGE_EXTENSIONS = {".jpg", ".jpeg", ".png", ".bmp", ".webp"}

def image_files_under(path):
    return sorted(
        file_path
        for file_path in path.rglob("*")
        if file_path.is_file()
        and file_path.suffix.lower() in IMAGE_EXTENSIONS
    )

test_class_dirs = sorted(
    folder
    for folder in EVAL_ROOT.iterdir()
    if folder.is_dir()
)

plantdoc_test_inventory = pd.DataFrame(
    {
        "plantdoc_folder": [folder.name for folder in test_class_dirs],
        "image_count": [
            len(image_files_under(folder))
            for folder in test_class_dirs
        ],
    }
).sort_values("plantdoc_folder").reset_index(drop=True)

print(f"PlantDoc test folders: {len(plantdoc_test_inventory)}")
print(f"PlantDoc test images: {plantdoc_test_inventory['image_count'].sum():,}")

display(plantdoc_test_inventory)

plantdoc_test_inventory.to_csv(
    RESULTS_DIR / "table_01_plantdoc_test_folder_inventory.csv",
    index=False,
)

In [ ]:
def find_files(root, filename):
    return sorted(
        file_path
        for file_path in root.rglob(filename)
        if file_path.is_file()
    )

CHECKPOINT_PATH = Path(
    "/kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/nb0_3/nb0_3/best_efficientnet_b0_main19.pt"
)

CLASS_MAPPING_PATH = Path(
    "/kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/metadata/class_to_index_main19.json"
)

INDEX_MAPPING_PATH = Path(
    "/kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/split_outputs/split_outputs/metadata/index_to_class_main19.json"
)

TEMPERATURE_METRICS_PATH = Path(
    "/kaggle/input/datasets/katakuricharlotte/crop-manifestfiles/nb_06/nb_06/proposed_model_temperature_scaling_metrics.csv"
)

print("\nSelected files:")
print("CHECKPOINT_PATH:", CHECKPOINT_PATH)
print("CLASS_MAPPING_PATH:", CLASS_MAPPING_PATH)
print("TEMPERATURE_PATH:", TEMPERATURE_PATH)

In [ ]:
with open(CLASS_MAPPING_PATH, "r", encoding="utf-8") as file:
    raw_mapping = json.load(file)

if all(str(key).isdigit() for key in raw_mapping.keys()):
    idx_to_class = {
        int(index): str(label)
        for index, label in raw_mapping.items()
    }

    class_to_idx = {
        label: index
        for index, label in idx_to_class.items()
    }
else:
    class_to_idx = {
        str(label): int(index)
        for label, index in raw_mapping.items()
    }

    idx_to_class = {
        index: label
        for label, index in class_to_idx.items()
    }

NUM_CLASSES = len(class_to_idx)

print(f"Number of model classes: {NUM_CLASSES}")
print(list(idx_to_class.items())[:10])

In [ ]:
# Notebook 10 — mappings only for classes actually used in this project.
#
# Every PlantDoc folder not used in the project is explicitly excluded.
# Cell 7 will verify that each non-None label exists in class_to_idx.json.
#
# IMPORTANT:
# Adjust only the right-side strings if Cell 7 shows that your class_to_idx.json
# uses a different exact spelling.

PLANTDOC_TO_PLANTVILLAGE = {
    # ---------------------------------------------------------
    # Corn / maize classes used in this project
    # ---------------------------------------------------------
    "Corn_Gray_leaf_spot": (
        "corn___cercospora_leaf_spot_gray_leaf_spot"
    ),
    "Corn_leaf_blight": "corn___northern_leaf_blight",
    "Corn_rust_leaf": "corn___common_rust",

    # PlantDoc test does not appear to contain a Corn healthy folder.
    # "Corn_leaf": "corn___healthy",

    # ---------------------------------------------------------
    # Bell pepper classes used in this project
    # ---------------------------------------------------------
    "Bell_pepper_leaf": "bell_pepper___healthy",
    "Bell_pepper_leaf_spot": "bell_pepper___bacterial_spot",

    # ---------------------------------------------------------
    # Potato classes used in this project
    # ---------------------------------------------------------
    "Potato_leaf_early_blight": "potato___early_blight",
    "Potato_leaf_late_blight": "potato___late_blight",

    # PlantDoc test does not appear to contain Potato healthy.
    # "Potato_leaf": "potato___healthy",

    # ---------------------------------------------------------
    # Tomato classes used in this project
    # ---------------------------------------------------------
    "Tomato_Early_blight_leaf": "tomato___early_blight",
    "Tomato_Septoria_leaf_spot": "tomato___septoria_leaf_spot",
    "Tomato_leaf": "tomato___healthy",
    "Tomato_leaf_bacterial_spot": "tomato___bacterial_spot",
    "Tomato_leaf_late_blight": "tomato___late_blight",
    "Tomato_leaf_mosaic_virus": "tomato___tomato_mosaic_virus",
    "Tomato_leaf_yellow_virus": (
        "tomato___tomato_yellow_leaf_curl_virus"
    ),
    "Tomato_mold_leaf": "tomato___leaf_mold",

    # Present in PlantDoc train; include automatically only if it
    # also occurs in the PlantDoc test directory.
    "Tomato_two_spotted_spider_mites_leaf": (
        "tomato___spider_mites_two_spotted_spider_mite"
    ),

    # PlantDoc test does not list tomato target spot.
    # "Tomato_Target_spot_leaf": "tomato___target_spot",

    # ---------------------------------------------------------
    # Explicitly excluded: outside this project's trained crop set
    # ---------------------------------------------------------
    "Apple_Scab_Leaf": None,
    "Apple_leaf": None,
    "Apple_rust_leaf": None,
    "Blueberry_leaf": None,
    "Cherry_leaf": None,
    "Peach_leaf": None,
    "Raspberry_leaf": None,
    "Soyabean_leaf": None,
    "Squash_Powdery_mildew_leaf": None,
    "Strawberry_leaf": None,
    "grape_leaf": None,
    "grape_leaf_black_rot": None,
}

In [ ]:
model_classes_df[
    model_classes_df["plantvillage_label"].str.contains(
        "corn|potato|tomato|bell_pepper",
        case=False,
        regex=True,
    )
]

In [ ]:
mapping_audit = plantdoc_test_inventory.copy()

mapping_audit["mapped_plantvillage_label"] = (
    mapping_audit["plantdoc_folder"]
    .map(PLANTDOC_TO_PLANTVILLAGE)
)

mapping_audit["mapping_status"] = np.select(
    [
        mapping_audit["plantdoc_folder"].map(
            PLANTDOC_TO_PLANTVILLAGE
        ).isna(),

        mapping_audit["mapped_plantvillage_label"].isin(
            class_to_idx.keys()
        ),
    ],
    [
        "excluded_no_semantic_match",
        "compatible",
    ],
    default="error_label_missing_from_model_mapping",
)

display(mapping_audit)

missing_model_labels = mapping_audit.loc[
    mapping_audit["mapping_status"]
    == "error_label_missing_from_model_mapping",
    ["plantdoc_folder", "mapped_plantvillage_label"],
]

if not missing_model_labels.empty:
    print(
        "The following mapped labels were not found in your "
        "class_to_idx.json:"
    )
    display(missing_model_labels)

    raise ValueError(
        "Update PLANTDOC_TO_PLANTVILLAGE using the exact labels displayed "
        "in Cell 5, then rerun Cells 6 and 7."
    )

compatible_mapping = mapping_audit.query(
    "mapping_status == 'compatible'"
).copy()

excluded_mapping = mapping_audit.query(
    "mapping_status != 'compatible'"
).copy()

print(f"Compatible folders: {len(compatible_mapping)}")
print(f"Excluded folders: {len(excluded_mapping)}")

print("\nCompatible labels:")
display(compatible_mapping)

print("\nExcluded labels:")
display(excluded_mapping)

mapping_audit.to_csv(
    RESULTS_DIR / "table_02_plantdoc_label_mapping_audit.csv",
    index=False,
)

In [ ]:
records = []

for _, mapping_row in compatible_mapping.iterrows():
    plantdoc_folder = mapping_row["plantdoc_folder"]
    true_label = mapping_row["mapped_plantvillage_label"]

    class_folder_path = EVAL_ROOT / plantdoc_folder
    class_image_paths = image_files_under(class_folder_path)

    crop, disease = true_label.split("___", maxsplit=1)

    for image_path in class_image_paths:
        records.append(
            {
                "image_path": str(image_path),
                "plantdoc_folder": plantdoc_folder,
                "true_label": true_label,
                "true_index": class_to_idx[true_label],
                "crop": crop,
                "disease": disease,
            }
        )

external_df = pd.DataFrame(records)

assert not external_df.empty, (
    "No compatible external images were found. "
    "Check the test root and mapping table."
)

external_class_summary = (
    external_df
    .groupby(
        ["plantdoc_folder", "true_label", "crop", "disease"],
        as_index=False,
    )
    .size()
    .rename(columns={"size": "images"})
    .sort_values(["crop", "disease"])
    .reset_index(drop=True)
)

print(f"Compatible PlantDoc test images: {len(external_df):,}")
print(
    "Compatible PlantDoc labels: "
    f"{external_df['true_label'].nunique()}"
)

display(external_class_summary)

external_df.to_csv(
    RESULTS_DIR / "plantdoc_test_compatible_manifest.csv",
    index=False,
)

external_class_summary.to_csv(
    RESULTS_DIR / "table_03_plantdoc_external_test_classes.csv",
    index=False,
)

In [ ]:
eval_transform = transforms.Compose(
    [
        transforms.Resize(256),
        transforms.CenterCrop(IMAGE_SIZE),
        transforms.ToTensor(),
        transforms.Normalize(
            mean=[0.485, 0.456, 0.406],
            std=[0.229, 0.224, 0.225],
        ),
    ]
)

class ExternalPlantDocDataset(Dataset):
    def __init__(self, dataframe, transform):
        self.dataframe = dataframe.reset_index(drop=True).copy()
        self.transform = transform

    def __len__(self):
        return len(self.dataframe)

    def __getitem__(self, index):
        row = self.dataframe.iloc[index]

        try:
            image = Image.open(row["image_path"]).convert("RGB")
            image = self.transform(image)
        except Exception as error:
            print(
                f"Unreadable image replaced with zeros: "
                f"{row['image_path']} | {error}"
            )
            image = torch.zeros(
                3,
                IMAGE_SIZE,
                IMAGE_SIZE,
                dtype=torch.float32,
            )

        return (
            image,
            int(row["true_index"]),
            row["image_path"],
            row["true_label"],
        )

external_dataset = ExternalPlantDocDataset(
    dataframe=external_df,
    transform=eval_transform,
)

external_loader = DataLoader(
    external_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=(DEVICE.type == "cuda"),
)

print(f"External test images: {len(external_dataset):,}")
print(f"Evaluation batches: {len(external_loader)}")

In [ ]:
def build_efficientnet_b0(num_classes):
    model = models.efficientnet_b0(weights=None)

    input_features = model.classifier[1].in_features

    model.classifier[1] = nn.Linear(
        input_features,
        num_classes,
    )

    return model

model = build_efficientnet_b0(NUM_CLASSES)

checkpoint = torch.load(
    CHECKPOINT_PATH,
    map_location="cpu",
)

if isinstance(checkpoint, dict):
    state_dict = (
        checkpoint.get("model_state_dict")
        or checkpoint.get("state_dict")
        or checkpoint.get("model")
        or checkpoint
    )
else:
    state_dict = checkpoint

state_dict = {
    key.replace("module.", ""): value
    for key, value in state_dict.items()
}

missing_keys, unexpected_keys = model.load_state_dict(
    state_dict,
    strict=False,
)

if missing_keys:
    print("Missing checkpoint keys:")
    print(missing_keys)

if unexpected_keys:
    print("Unexpected checkpoint keys:")
    print(unexpected_keys)

model = model.to(DEVICE)
model.eval()

temperature = 1.0

if TEMPERATURE_PATH is not None and TEMPERATURE_PATH.exists():
    with open(TEMPERATURE_PATH, "r", encoding="utf-8") as file:
        temperature_data = json.load(file)

    temperature = float(
        temperature_data.get(
            "temperature",
            temperature_data.get("T", 1.0),
        )
    )

print("Model: EfficientNet-B0")
print(f"Number of model classes: {NUM_CLASSES}")
print(f"Confidence temperature: {temperature:.4f}")

In [ ]:
@torch.inference_mode()
def predict_external_dataset(model, loader, temperature=1.0):
    prediction_rows = []

    for images, targets, image_paths, true_labels in loader:
        images = images.to(
            DEVICE,
            non_blocking=(DEVICE.type == "cuda"),
        )

        logits = model(images)

        calibrated_logits = logits / max(float(temperature), 1e-6)

        probabilities = torch.softmax(
            calibrated_logits,
            dim=1,
        )

        confidence, predicted_indices = probabilities.max(dim=1)

        top5_probabilities, top5_indices = torch.topk(
            probabilities,
            k=min(5, probabilities.shape[1]),
            dim=1,
        )

        for batch_index in range(images.shape[0]):
            true_index = int(targets[batch_index].item())
            predicted_index = int(
                predicted_indices[batch_index].item()
            )

            top5_indices_item = (
                top5_indices[batch_index]
                .detach()
                .cpu()
                .tolist()
            )

            top5_probabilities_item = (
                top5_probabilities[batch_index]
                .detach()
                .cpu()
                .tolist()
            )

            prediction_rows.append(
                {
                    "image_path": image_paths[batch_index],
                    "true_label": true_labels[batch_index],
                    "true_index": true_index,
                    "predicted_label": idx_to_class[
                        predicted_index
                    ],
                    "predicted_index": predicted_index,
                    "confidence": float(
                        confidence[batch_index].item()
                    ),
                    "top5_labels": json.dumps(
                        [
                            idx_to_class[int(class_index)]
                            for class_index in top5_indices_item
                        ]
                    ),
                    "top5_probabilities": json.dumps(
                        [
                            float(probability)
                            for probability
                            in top5_probabilities_item
                        ]
                    ),
                    "correct_top1": int(
                        predicted_index == true_index
                    ),
                    "correct_top5": int(
                        true_index in top5_indices_item
                    ),
                }
            )

    return pd.DataFrame(prediction_rows)

start_time = time.time()

predictions_df = predict_external_dataset(
    model=model,
    loader=external_loader,
    temperature=temperature,
)

elapsed_seconds = time.time() - start_time

predictions_df = predictions_df.merge(
    external_df[
        [
            "image_path",
            "plantdoc_folder",
            "crop",
            "disease",
        ]
    ],
    on="image_path",
    how="left",
)

print(
    f"External inference completed in "
    f"{elapsed_seconds:.1f} seconds."
)

display(predictions_df.head())

predictions_df.to_csv(
    RESULTS_DIR / "plantdoc_zero_shot_predictions.csv",
    index=False,
)

In [ ]:
y_true = predictions_df["true_index"].to_numpy()
y_pred = predictions_df["predicted_index"].to_numpy()

overall_metrics = pd.DataFrame(
    [
        {
            "evaluation_domain": (
                "PlantDoc test external zero-shot"
            ),
            "model": "EfficientNet-B0",
            "images": len(predictions_df),
            "compatible_true_classes": predictions_df[
                "true_label"
            ].nunique(),
            "top1_accuracy": accuracy_score(y_true, y_pred),
            "balanced_accuracy": balanced_accuracy_score(
                y_true,
                y_pred,
            ),
            "macro_f1": f1_score(
                y_true,
                y_pred,
                average="macro",
                zero_division=0,
            ),
            "weighted_f1": f1_score(
                y_true,
                y_pred,
                average="weighted",
                zero_division=0,
            ),
            "top5_accuracy": predictions_df[
                "correct_top5"
            ].mean(),
            "mean_confidence": predictions_df[
                "confidence"
            ].mean(),
            "median_confidence": predictions_df[
                "confidence"
            ].median(),
            "accuracy_at_all_predictions": predictions_df[
                "correct_top1"
            ].mean(),
            "inference_seconds": elapsed_seconds,
        }
    ]
)

display(overall_metrics.T)

overall_metrics.to_csv(
    RESULTS_DIR / "table_04_external_domain_metrics.csv",
    index=False,
)

In [ ]:
evaluated_indices = sorted(
    predictions_df["true_index"].unique()
)

evaluated_labels = [
    idx_to_class[class_index]
    for class_index in evaluated_indices
]

report = classification_report(
    y_true,
    y_pred,
    labels=evaluated_indices,
    target_names=evaluated_labels,
    output_dict=True,
    zero_division=0,
)

per_class_metrics = (
    pd.DataFrame(report)
    .T
    .reset_index()
    .rename(columns={"index": "true_label"})
)

per_class_metrics = per_class_metrics.loc[
    ~per_class_metrics["true_label"].isin(
        ["accuracy", "macro avg", "weighted avg"]
    )
].copy()

support_by_class = (
    predictions_df
    .groupby("true_label")
    .size()
    .reset_index(name="images")
)

per_class_metrics = (
    per_class_metrics
    .merge(
        support_by_class,
        on="true_label",
        how="left",
    )
    .rename(
        columns={
            "f1-score": "f1_score",
            "precision": "precision",
            "recall": "recall",
        }
    )
    .sort_values(
        ["f1_score", "images"],
        ascending=[True, False],
    )
    .reset_index(drop=True)
)

display(per_class_metrics)

per_class_metrics.to_csv(
    RESULTS_DIR / "table_05_external_per_class_metrics.csv",
    index=False,
)

In [ ]:
confusion = confusion_matrix(
    y_true,
    y_pred,
    labels=evaluated_indices,
    normalize="true",
)

figure_size = max(12, len(evaluated_labels) * 0.60)

plt.figure(figsize=(figure_size, figure_size * 0.85))

sns.heatmap(
    confusion,
    cmap="Blues",
    xticklabels=evaluated_labels,
    yticklabels=evaluated_labels,
    vmin=0,
    vmax=1,
    square=True,
    cbar_kws={
        "label": "Row-normalized proportion"
    },
)

plt.title(
    "PlantDoc external-domain confusion matrix\n"
    "Rows = true PlantDoc-mapped class; columns = prediction"
)

plt.xlabel("Predicted PlantVillage class")
plt.ylabel("True PlantDoc-mapped class")

plt.xticks(rotation=90, fontsize=8)
plt.yticks(rotation=0, fontsize=8)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fig_01_plantdoc_confusion_matrix.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

In [ ]:
confidence_plot_df = predictions_df.copy()

confidence_plot_df["prediction_status"] = np.where(
    confidence_plot_df["correct_top1"].eq(1),
    "Correct",
    "Incorrect",
)

confidence_summary = (
    confidence_plot_df
    .groupby("prediction_status", as_index=False)
    .agg(
        images=("image_path", "count"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
        accuracy=("correct_top1", "mean"),
    )
)

display(confidence_summary)

confidence_summary.to_csv(
    RESULTS_DIR / "table_06_confidence_by_correctness.csv",
    index=False,
)

plt.figure(figsize=(9, 5))

sns.histplot(
    data=confidence_plot_df,
    x="confidence",
    hue="prediction_status",
    bins=25,
    stat="density",
    common_norm=False,
    element="step",
)

plt.title(
    "PlantDoc confidence distribution: "
    "correct versus incorrect predictions"
)

plt.xlabel("Top-1 calibrated confidence")
plt.ylabel("Density")
plt.xlim(0, 1)

plt.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "fig_02_plantdoc_confidence_correct_vs_incorrect.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

In [ ]:
confidence_thresholds = np.round(
    np.arange(0.00, 1.01, 0.05),
    2,
)

selective_rows = []

for threshold in confidence_thresholds:
    retained_df = predictions_df.loc[
        predictions_df["confidence"] >= threshold
    ].copy()

    coverage = len(retained_df) / len(predictions_df)

    if len(retained_df) > 0:
        selective_accuracy = retained_df[
            "correct_top1"
        ].mean()
    else:
        selective_accuracy = np.nan

    if retained_df["true_index"].nunique() > 1:
        selective_macro_f1 = f1_score(
            retained_df["true_index"],
            retained_df["predicted_index"],
            average="macro",
            zero_division=0,
        )
    else:
        selective_macro_f1 = np.nan

    selective_rows.append(
        {
            "confidence_threshold": threshold,
            "retained_images": len(retained_df),
            "coverage": coverage,
            "selective_accuracy": selective_accuracy,
            "selective_macro_f1": selective_macro_f1,
        }
    )

selective_df = pd.DataFrame(selective_rows)

display(selective_df)

selective_df.to_csv(
    RESULTS_DIR
    / "table_07_selective_prediction_by_threshold.csv",
    index=False,
)

fig, axis_coverage = plt.subplots(figsize=(9, 5))

axis_coverage.plot(
    selective_df["confidence_threshold"],
    selective_df["coverage"],
    marker="o",
    color="#1f77b4",
)

axis_coverage.set_xlabel("Confidence threshold")
axis_coverage.set_ylabel(
    "Prediction coverage",
    color="#1f77b4",
)

axis_coverage.tick_params(
    axis="y",
    labelcolor="#1f77b4",
)

axis_coverage.set_ylim(0, 1.05)

axis_accuracy = axis_coverage.twinx()

axis_accuracy.plot(
    selective_df["confidence_threshold"],
    selective_df["selective_accuracy"],
    marker="s",
    color="#d62728",
)

axis_accuracy.set_ylabel(
    "Accuracy among retained predictions",
    color="#d62728",
)

axis_accuracy.tick_params(
    axis="y",
    labelcolor="#d62728",
)

axis_accuracy.set_ylim(0, 1.05)

plt.title(
    "PlantDoc selective prediction: "
    "coverage versus retained accuracy"
)

fig.tight_layout()

plt.savefig(
    RESULTS_DIR
    / "fig_03_plantdoc_selective_prediction.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

In [ ]:
crop_metrics = (
    predictions_df
    .groupby("crop", as_index=False)
    .agg(
        images=("image_path", "count"),
        compatible_classes=("true_label", "nunique"),
        top1_accuracy=("correct_top1", "mean"),
        mean_confidence=("confidence", "mean"),
        median_confidence=("confidence", "median"),
    )
    .sort_values("top1_accuracy")
    .reset_index(drop=True)
)

display(crop_metrics)

crop_metrics.to_csv(
    RESULTS_DIR / "table_08_external_crop_metrics.csv",
    index=False,
)

plt.figure(figsize=(10, 5))

sns.barplot(
    data=crop_metrics,
    x="top1_accuracy",
    y="crop",
    palette="viridis",
)

plt.xlim(0, 1)
plt.xlabel("Top-1 accuracy")
plt.ylabel("Crop")
plt.title("PlantDoc zero-shot external performance by crop")

plt.tight_layout()

plt.savefig(
    RESULTS_DIR / "fig_04_plantdoc_accuracy_by_crop.png",
    dpi=250,
    bbox_inches="tight",
)

plt.show()

In [ ]:
high_confidence_errors = (
    predictions_df
    .loc[predictions_df["correct_top1"] == 0]
    .sort_values("confidence", ascending=False)
    .reset_index(drop=True)
)

display(
    high_confidence_errors[
        [
            "image_path",
            "plantdoc_folder",
            "true_label",
            "predicted_label",
            "confidence",
            "top5_labels",
            "top5_probabilities",
        ]
    ].head(30)
)

high_confidence_errors.to_csv(
    RESULTS_DIR / "table_09_high_confidence_external_errors.csv",
    index=False,
)

In [ ]:
def show_prediction_grid(
    dataframe,
    n=12,
    title="PlantDoc external prediction errors",
):
    subset = dataframe.head(n).copy()

    if subset.empty:
        print("No examples available.")
        return

    columns = 4
    rows = int(np.ceil(len(subset) / columns))

    figure, axes = plt.subplots(
        rows,
        columns,
        figsize=(16, 4 * rows),
    )

    axes = np.asarray(axes).reshape(-1)

    for axis, (_, row) in zip(axes, subset.iterrows()):
        try:
            image = Image.open(
                row["image_path"]
            ).convert("RGB")

            axis.imshow(image)

        except Exception:
            axis.text(
                0.5,
                0.5,
                "Image unavailable",
                ha="center",
                va="center",
            )

        axis.set_title(
            f"True: {row['true_label']}\n"
            f"Predicted: {row['predicted_label']}\n"
            f"Confidence: {row['confidence']:.1%}",
            fontsize=9,
        )

        axis.axis("off")

    for axis in axes[len(subset):]:
        axis.axis("off")

    figure.suptitle(
        title,
        fontsize=15,
        y=1.01,
    )

    plt.tight_layout()
    plt.show()

show_prediction_grid(
    dataframe=high_confidence_errors,
    n=12,
    title=(
        "Highest-confidence PlantDoc errors: "
        "qualitative evidence of domain shift"
    ),
)

In [ ]:
run_metadata = {
    "notebook": "10_external_domain_shift_plantdoc",
    "evaluation_protocol": (
        "Zero-shot evaluation on PlantDoc test split only"
    ),
    "model": "EfficientNet-B0",
    "checkpoint_path": str(CHECKPOINT_PATH),
    "class_mapping_path": str(CLASS_MAPPING_PATH),
    "plantdoc_root": str(PLANTDOC_ROOT),
    "plantdoc_test_root": str(EVAL_ROOT),
    "image_size": IMAGE_SIZE,
    "batch_size": BATCH_SIZE,
    "device": str(DEVICE),
    "temperature": float(temperature),
    "external_images_evaluated": int(len(predictions_df)),
    "compatible_external_classes": int(
        predictions_df["true_label"].nunique()
    ),
    "top1_accuracy": float(
        overall_metrics.loc[0, "top1_accuracy"]
    ),
    "balanced_accuracy": float(
        overall_metrics.loc[0, "balanced_accuracy"]
    ),
    "macro_f1": float(
        overall_metrics.loc[0, "macro_f1"]
    ),
    "weighted_f1": float(
        overall_metrics.loc[0, "weighted_f1"]
    ),
    "top5_accuracy": float(
        overall_metrics.loc[0, "top5_accuracy"]
    ),
    "excluded_plantdoc_folders": excluded_mapping[
        "plantdoc_folder"
    ].tolist(),
    "notes": (
        "PlantDoc train images were not used. Only semantically "
        "compatible PlantDoc test labels were included in supervised "
        "performance metrics. Results quantify external domain shift."
    ),
}

with open(
    RESULTS_DIR / "run_metadata.json",
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        run_metadata,
        file,
        indent=2,
    )

print(json.dumps(run_metadata, indent=2))

## Interpretation and reporting guidance

This experiment is a zero-shot external-domain evaluation. PlantDoc images differ from PlantVillage through background clutter, illumination changes, variable camera angle, multiple leaves, occlusion, and varied disease presentation.

Report:

- Compatible PlantDoc test image count and class count.
- Top-1 accuracy, balanced accuracy, macro F1, weighted F1, and top-5 accuracy.
- Per-class recall and F1 score.
- Confidence distributions for correct and incorrect predictions.
- Coverage-versus-accuracy behavior under confidence thresholds.
- Representative high-confidence errors and their visual causes.

Do not claim that PlantDoc test performance is directly comparable to the PlantVillage random-split test score. A drop in performance is expected and is evidence of domain shift.

Suggested report sentence:

> “The final PlantVillage-trained EfficientNet-B0 model was evaluated zero-shot on semantically compatible PlantDoc test classes. Performance decreased under external-domain conditions, reflecting the effect of field backgrounds, illumination changes, clutter, occlusion, and symptom-appearance variation. Confidence-based selective prediction improved reliability among retained predictions but did not remove domain-shift risk.”